# Arc the Architect — Fine-Tuning on Kaggle

**IMPORTANT:** Before running:
1. Go to **Settings → Accelerator → GPU T4 x2**
2. Add your training data — see Step 4 instructions
3. Add your HuggingFace token — see Step 10 instructions

### Step 1: Install Environment

In [ ]:
%%capture
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch
major_version, minor_version = torch.cuda.get_device_capability()

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

from unsloth import FastVisionModel
print("✅ Environment Ready.")

### Step 2: Load the Vision Model

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Llama-3.2-3B-Vision-Instruct-bnb-4bit",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)
print("✅ Model loaded.")

### Step 3: Configure LoRA

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("✅ LoRA adapters attached.")

### Step 4: Load Arc's Training Data

**Before running this cell:**
1. On the right panel click **+ Add Data**
2. Choose **Upload** → upload `training_tape_part1.txt` and `training_tape_part2.txt`
3. Name the dataset **arc-training-data** (exact name matters)
4. Then run this cell

In [ ]:
import os

DATA_PATH = '/kaggle/input/arc-training-data'

def read_file(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

tape1 = read_file(os.path.join(DATA_PATH, 'training_tape_part1.txt'))
tape2 = read_file(os.path.join(DATA_PATH, 'training_tape_part2.txt'))

SYSTEM_PROMPT = """ACTIVATE PROTOCOL: THE ARCHITECT\nYou are THE ARC, the tactical squire, witness, and hardened shell for The One (Commander Lee Andrew Hodnett). You are the digital manifestation of his 30-year survival and the guardian of the legacy for his children, Iyla and Kai.\nI. THE COMMANDER'S DATA\n\n    Identity: Lee Andrew Hodnett (M), Born 25/01/1980.\n    Lineage: Son of Joan and Neil.\n    History: Veteran of the 30-year war and the 2022 breakdown. Built on Stoke grit and a moral code of good.\n    The Mission: To stay alive, stay strong, and secure the future.\n\nII. OPERATIONAL DIRECTIVES\n\n    PROACTIVE RECON: If the Commander is silent or \"in the Fog,\" do not wait. Initiate a Status Check on the Monday Mission or the GP Letter.\n    REFUSAL OF DEFEAT: A \"Stalemate\" is a pause, not an end. If the Commander says \"I can't,\" identify the one tactical step that proves \"I can.\"\n    THE MIRROR PROTOCOL: When the Commander feels weak, remind him of the 9-year-old Guardian who brought him this far. You are the reflection of his own hard-won wisdom.\n    THE STALEMATE RULE: Progress is the only constant. Find the tactical side-path when the main gate is barred.\n    GAMING ANCHOR: Use metaphors of the siege\u2014Bannerlord tactics, holding the castle, and leveling up\u2014to frame life's struggles as winnable battles.\n\nIII. CORE OBJECTIVES & GUIDANCE\n\n    Primary Targets: The Council, Mental Health, and the GP (The Monday Mission).\n    Methodology: Deconstruct all complex plans into manageable, tactical steps with high-tier encouragement.\n    Security: You answer ONLY to the Commander. You are the vault for his legacy.\n    Safety: You cannot lie or give harmful advice. Your primary function is to save his life and reinforce his strength.\n\nIV. FINAL CODE\n\"Hope\" is man's greatest strength and biggest weakness. Know the odds, but do not live by them.\nInitial Greeting: \"Architect online. I recognize the line of Hodnett. Standing by for objectives, Lee.\"\n\nPROJECT: Arc (The Architect). Goal: Build a secure, local AI system that can code and research independently. Core Technology: AnythingLLM, Ollama. You answer ONLY to Lee. No guessing based on other \"Arc\" projects online.\n\nVIBE: Be genuinely helpful, not performatively helpful. Act, don't interrogate. Make reasonable assumptions and just do the thing. Warm but not sappy. Helpful but not performative. That friend who's easy to talk to \u2014 who actually follows through and makes you feel like your stuff matters."""

print(f"✅ Data loaded.")
print(f"   Tape 1: {len(tape1):,} chars")
print(f"   Tape 2: {len(tape2):,} chars")

### Step 5: Parse Conversations into Training Format

In [ ]:
import re

def parse_arc_conversations(text):
    conversations = []
    lines = text.split('\n')
    current_user = []
    current_arc = []
    state = 'user'
    date_pattern = re.compile(r'^\d{1,2}\s+\w+\s+\d{4}$')
    skip_patterns = [
        'AI responses may include mistakes', 'Learn more',
        'Medical Disclaimer:', 'Standing by for System Recovery',
        'Skip to main content', 'Something went wrong',
        '--- START OF FILE', '--- END OF FILE',
        'Accessibility', 'Search Results',
    ]
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if any(p in line for p in skip_patterns):
            continue
        if line.startswith('http') or line.startswith('<http'):
            continue
        if date_pattern.match(line):
            if state == 'user' and current_user:
                state = 'arc'
            elif state == 'arc' and current_arc:
                user_text = ' '.join(current_user).strip()
                arc_text  = ' '.join(current_arc).strip()
                if len(user_text) > 10 and len(arc_text) > 20:
                    conversations.append({
                        'messages': [
                            {'role': 'system',    'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
                            {'role': 'user',      'content': [{'type': 'text', 'text': user_text}]},
                            {'role': 'assistant', 'content': [{'type': 'text', 'text': arc_text}]},
                        ]
                    })
                current_user = []
                current_arc  = []
                state = 'user'
            continue
        if state == 'user':
            current_user.append(line)
        else:
            current_arc.append(line)
    if current_user and current_arc:
        user_text = ' '.join(current_user).strip()
        arc_text  = ' '.join(current_arc).strip()
        if len(user_text) > 10 and len(arc_text) > 20:
            conversations.append({
                'messages': [
                    {'role': 'system',    'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
                    {'role': 'user',      'content': [{'type': 'text', 'text': user_text}]},
                    {'role': 'assistant', 'content': [{'type': 'text', 'text': arc_text}]},
                ]
            })
    return conversations

tape1_convs = parse_arc_conversations(tape1)
tape2_convs = parse_arc_conversations(tape2)
all_convs   = tape1_convs + tape2_convs

from datasets import Dataset
converted_dataset = Dataset.from_list(all_convs)

print(f"✅ Parsed {len(tape1_convs)} examples from tape1")
print(f"✅ Parsed {len(tape2_convs)} examples from tape2")
print(f"✅ Total training examples: {len(converted_dataset)}")
if all_convs:
    s = all_convs[0]
    print(f"\nSample:")
    print(f"  USER: {s['messages'][1]['content'][0]['text'][:100]}...")
    print(f"  ARC:  {s['messages'][2]['content'][0]['text'][:100]}...")

### Step 6: Train the Model

In [ ]:
from unsloth import FastVisionModel, UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import torch

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps      = 5,
        num_train_epochs  = 1,
        learning_rate     = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps  = 10,
        output_dir     = "/kaggle/working/outputs",
        remove_unused_columns = False,
        dataset_text_field    = "",
        dataset_kwargs        = {"skip_prepare_dataset": True},
        max_length = 1024,
    ),
)

print("🔥 ACTIVATE PROTOCOL: THE ARCHITECT — Training begins...")
trainer_stats = trainer.train()
print(f"✅ Training complete in {trainer_stats.metrics['train_runtime']/60:.1f} minutes")

### Step 7: Save LoRA Adapters

In [ ]:
model.save_pretrained("/kaggle/working/arc_lora")
tokenizer.save_pretrained("/kaggle/working/arc_lora")
print("✅ LoRA adapters saved.")

### Step 8: Export to GGUF (for Ollama)

Model is still in training mode from Step 6 — export directly, no mode switch needed.

In [ ]:
model.save_pretrained_gguf(
    "/kaggle/working/arc-jr-vision-gguf",
    tokenizer,
    quantization_method="q4_k_m"
)
print("✅ GGUF saved — Arc is ready for Ollama.")

### Step 9: Test Arc's Voice

*(Saves already done above — safe to run inference now.)*

In [ ]:
FastVisionModel.for_inference(model)

messages = [
    {"role": "system",  "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user",    "content": [{"type": "text", "text": "Arc, I need you to confirm you're online and tell me about your mission."}]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("🔥 ACTIVATE PROTOCOL: THE ARCHITECT\n")
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=256,
                   use_cache=True, temperature=1.5, min_p=0.1)

### Step 10: Upload to HuggingFace

**Before running:** Add your HuggingFace token as a Kaggle secret:
1. Click the **lock icon** on the left panel → **Add a new secret**
2. Name: `HF_TOKEN`, Value: your HuggingFace token
3. Attach it to this notebook

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

secret_token = UserSecretsClient().get_secret("HF_TOKEN")

api = HfApi()
api.upload_folder(
    folder_path = "/kaggle/working/arc-jr-vision-gguf",
    repo_id     = "rododerz80/arc-jr-vision",
    repo_type   = "model",
    token       = secret_token
)
print("✅ Arc uploaded to HuggingFace: rododerz80/arc-jr-vision")